Copyright 2026, Battelle Energy Alliance, LLC, ALL RIGHTS RESERVED

In [2]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os
os.chdir("../../..")
from HelpingFunctions import ERCOTProcessor
from HelpingFunctions import WeatherProcessing
from HelpingFunctions import FeatureEngineering
from HelpingFunctions import ForecastingHelpers

import onnxruntime as ort
ort.set_default_logger_severity(4)

In [3]:
import os 
os.getcwd()

'/home/ortild/Amaranth/opensourcegridmodeling'

Data Preprocessing for RT Data - Setting Up Model Inputs

In [4]:
full_df = pd.read_csv('ElectricityDemandAustinTX/LoadForecastingAttacks/full_data.csv', parse_dates=['time'], index_col=['time'])
hourly_res_norm = ForecastingHelpers.hourlyresiduals(full_df)

/home/ortild/Amaranth/opensourcegridmodeling/ElectricityDemandAustinTX/LoadForecastingAttacks/HelpingFunctions/ForecastingHelpers.py:74: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  hourly_res_norm['load'] = df_norm['load'].groupby(pd.Grouper(freq='M')).transform(lambda x: x - x.mean())


In [5]:
# train-validate-test split
train = hourly_res_norm[:'2014']
validate = hourly_res_norm['2015':'2016']
test = hourly_res_norm['2017':]

# setup training variables 
exog_tr = train.iloc[:,1:].values
ar_tr = train['load'].shift().bfill().values[:,None]
X_tr = np.hstack([ar_tr, exog_tr])
y_tr = train['load'].values

# setup validation variables
exog_val = validate.iloc[:,1:].values
y_val = validate['load'].values

# setup testing variables
exog_te = test.iloc[:,1:].values
ar_test = test['load'].shift().bfill().values[:,None]
y_test = test['load'].values
X_test = np.hstack([ar_test, exog_te])

# setup miscellaneous variables
yp_full = hourly_res_norm.loc[:'2016','load']
yp_val = hourly_res_norm.loc['2015':'2016','load']
yp_te = hourly_res_norm.loc['2017':,'load']
y_init_val = np.hstack([y_tr[-1], validate.iloc[167::168,0].values])
y_init_te = np.hstack([y_val[-1], test.iloc[167::168,0].values])

Linear Regression Model

In [6]:
import onnx
onnx_model =  onnx.load("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/linear_no_mapie.onnx")

In [7]:
graph = onnx_model.graph
#print(onnx_model)
print(graph)
weights = onnx_model.graph.initializer[0].ListFields()[3][1][:]
interceptList = onnx_model.graph.initializer[1].ListFields()[3][1][:]

node {
  input: "X"
  input: "coef"
  output: "multiplied"
  name: "MatMul"
  op_type: "MatMul"
  domain: ""
}
node {
  input: "multiplied"
  input: "intercept"
  output: "resh"
  name: "Add"
  op_type: "Add"
  domain: ""
}
node {
  input: "resh"
  input: "shape_tensor"
  output: "variable"
  name: "Reshape"
  op_type: "Reshape"
  domain: ""
}
name: "ONNX(LinearRegression)"
initializer {
  dims: 13
  dims: 1
  data_type: 11
  name: "coef"
  double_data: 0.938745416511678
  double_data: -0.0029355835524287367
  double_data: -6.0131328821698735e-05
  double_data: -0.0017936289492126256
  double_data: -6.5064154666689841e-05
  double_data: -0.000911993588956408
  double_data: -0.0014550393514231315
  double_data: -0.0012154378839187086
  double_data: -5.2041704279304213e-18
  double_data: 0.0036475349789651021
  double_data: 0.0019535461581610255
  double_data: 0.0027213955768126121
  double_data: -0.022888753747047845
}
initializer {
  dims: 1
  data_type: 11
  name: "intercept"
  double

The onnx_model graph explains the operations performed on the inputs to receive the model output. 
The first node describes taking an input, X, and multiplying X with the model coefficients, coef, to produce a node output called multiplied. 
The second node describes taking multiplied and adding an intercept, intercept, to produce a node output called resh.
The third node describes reshaping the addition output to a single value for the output.

In [8]:
manualX = X_test[0]
coef = np.array(weights)
multiplied = manualX * coef
print(multiplied)
intercept = interceptList[0]
print(intercept)
resh = multiplied + intercept
print(resh)
o = multiplied.sum() + intercept
print(o)
manualY = y_test[0]
print(manualY)

[-9.31674198e-02 -1.42331324e-03 -5.59221358e-05 -0.00000000e+00
 -0.00000000e+00 -0.00000000e+00 -1.45503935e-03 -0.00000000e+00
 -0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
 -2.28887537e-02]
0.001777452248230074
[-0.09138997  0.00035414  0.00172153  0.00177745  0.00177745  0.00177745
  0.00032241  0.00177745  0.00177745  0.00177745  0.00177745  0.00177745
 -0.0211113 ]
-0.11721299598576972
-0.09924673731924893


In [9]:
manualX = X_test
coef = np.array(weights)
multiplied = manualX * coef
print(multiplied)
intercept = interceptList[0]
resh = multiplied + intercept
print(resh)
output = resh.reshape(-1)
print(output)
print()
manualY = y_test[0]
print(manualY)

[[-9.31674198e-02 -1.42331324e-03 -5.59221358e-05 ...  0.00000000e+00
   0.00000000e+00 -2.28887537e-02]
 [-9.31674198e-02 -1.42331324e-03 -5.59221358e-05 ...  0.00000000e+00
   7.04349005e-04 -2.21088384e-02]
 [-8.96179405e-02 -1.42331324e-03 -5.59221358e-05 ...  0.00000000e+00
   1.36069779e-03 -1.98222422e-02]
 ...
 [-2.11271508e-03 -1.68425400e-03 -5.37574080e-05 ...  1.95354616e-03
  -1.92431727e-03 -1.61847930e-02]
 [-1.63671447e-02 -1.68425400e-03 -5.37574080e-05 ...  1.95354616e-03
  -1.36069779e-03 -1.98222422e-02]
 [-2.91316343e-02 -1.68425400e-03 -5.37574080e-05 ...  1.95354616e-03
  -7.04349005e-04 -2.21088384e-02]]
[[-9.13899675e-02  3.54139011e-04  1.72153011e-03 ...  1.77745225e-03
   1.77745225e-03 -2.11113015e-02]
 [-9.13899675e-02  3.54139011e-04  1.72153011e-03 ...  1.77745225e-03
   2.48180125e-03 -2.03313861e-02]
 [-8.78404883e-02  3.54139011e-04  1.72153011e-03 ...  1.77745225e-03
   3.13815004e-03 -1.80447900e-02]
 ...
 [-3.35262829e-04  9.31982505e-05  1.7236948

In [10]:
inputsL = manualX
print(inputsL)

[[-0.09924674  0.48484848  0.93       ...  0.          0.
   1.        ]
 [-0.09924674  0.48484848  0.93       ...  0.          0.25881905
   0.96592583]
 [-0.09546565  0.48484848  0.93       ...  0.          0.5
   0.8660254 ]
 ...
 [-0.00225057  0.57373737  0.894      ...  1.         -0.70710678
   0.70710678]
 [-0.01743513  0.57373737  0.894      ...  1.         -0.5
   0.8660254 ]
 [-0.03103252  0.57373737  0.894      ...  1.         -0.25881905
   0.96592583]]


In [11]:
import numpy as np
from sklearn.linear_model import LinearRegression
coef = np.array(weights)
intercept = interceptList[0]
model = LinearRegression(fit_intercept=False)
model.coef_ = coef
model.intercept_ = intercept
manualX = np.array(X_test)
predictions = model.predict(manualX)

print(f"Coefficients: {model.coef_}")
print(f"Intercept: {model.intercept_}")
print(f"Predictions: {predictions}")

Coefficients: [ 9.38745417e-01 -2.93558355e-03 -6.01313288e-05 -1.79362895e-03
 -6.50641547e-05 -9.11993589e-04 -1.45503935e-03 -1.21543788e-03
 -5.20417043e-18  3.64753498e-03  1.95354616e-03  2.72139558e-03
 -2.28887537e-02]
Intercept: 0.001777452248230074
Predictions: [-0.117213   -0.11572873 -0.10967113 ... -0.0201042  -0.03743246
 -0.0518272 ]


In [12]:
import onnxruntime as rt
model_inv = onnx_model
session = rt.InferenceSession("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/linear_no_mapie.onnx")
input_name = session.get_inputs()[0].name
input_data = np.array(inputsL, dtype=np.float64) 
#input_data = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
inputs = {input_name: input_data}
outputs = session.run(None, inputs)
print(outputs)

[array([[-0.117213  ],
       [-0.11572873],
       [-0.10967113],
       ...,
       [-0.0201042 ],
       [-0.03743246],
       [-0.0518272 ]], shape=(70128, 1))]


In [13]:
from sklearn.metrics import mean_absolute_error
old_out = []
for item in outputs[0]:
    old_out.append(item[0])
print(mean_absolute_error(old_out,predictions))

5.4430511119297944e-18
